# 🔍 RAG Saturation & Hallucination Analysis
## Kapita Selekta SC — Final Assignment
### Dataset: IDK-MRC | Model: Qwen2.5-1.5B-Instruct (4-bit) | Retriever: FAISS

**Deskripsi:**  
Notebook ini mengimplementasikan sistem RAG lengkap untuk menginvestigasi:
1. **Saturasi performa** — apakah kualitas jawaban terus meningkat seiring bertambahnya K?
2. **Halusinasi sitasi** — seberapa sering LLM mengutip dokumen yang salah?

**Eksperimen:** K ∈ {1, 3, 5, 10} diuji menggunakan metrik ROUGE, BLEU, dan BERTScore.

## ⚙️ Cell 1 — Install Dependencies

In [ ]:
# ============================================================
# CELL 1: Install semua dependensi yang dibutuhkan
# Runtime: ~2-3 menit pada Google Colab
# ============================================================
!pip install -q datasets faiss-cpu sentence-transformers
!pip install -q -U transformers accelerate bitsandbytes
!pip install -q rouge-score bert-score nltk
!pip install -q matplotlib seaborn pandas tqdm openpyxl


import nltk
nltk.download('punkt', quiet=True)
nltk.download('punkt_tab', quiet=True)

print("✅ Semua dependensi berhasil diinstall!")


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.2/18.2 MB 62.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.8/10.8 MB 38.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 10.4 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.1/61.1 kB 4.4 MB/s eta 0:00:00
✅ Semua dependensi berhasil diinstall!


## 📦 Cell 2 — Import Libraries

In [ ]:
# ============================================================
# CELL 2: Import semua library yang diperlukan
# ============================================================
import os
import gc
import re
import json
import time
import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import seaborn as sns
from tqdm.notebook import tqdm
from collections import defaultdict

import torch
print(f"🖥️  PyTorch version : {torch.__version__}")
print(f"🖥️  CUDA available  : {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"🖥️  GPU             : {torch.cuda.get_device_name(0)}")
    print(f"🖥️  VRAM            : {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")

from datasets import load_dataset
import faiss
from sentence_transformers import SentenceTransformer

from transformers import (
    AutoTokenizer,
    AutoModelForCausalLM,
    BitsAndBytesConfig,
)

from rouge_score import rouge_scorer
from bert_score import score as bert_score_fn
from nltk.translate.bleu_score import sentence_bleu, SmoothingFunction, corpus_bleu

# Reproducibility
np.random.seed(42)
torch.manual_seed(42)

print("\n✅ Semua library berhasil diimport!")


🖥️  PyTorch version : 2.10.0+cu128
🖥️  CUDA available  : True
🖥️  GPU             : Tesla T4
🖥️  VRAM            : 15.6 GB

✅ Semua library berhasil diimport!


## 📂 Cell 3 — Load & Eksplorasi Dataset IDK-MRC

In [ ]:
# ============================================================
# CELL 3: Load dataset IDK-MRC dari HuggingFace
# IDK-MRC = Indonesian Knowledge Machine Reading Comprehension
# Format: setiap item memiliki 'context' dan 'qas' (daftar QA)
# ============================================================
print("⏳ Memuat dataset IDK-MRC...")
ds = load_dataset("rifkiaputri/idk-mrc")
print("\n✅ Dataset berhasil dimuat!")
print(f"\n{ds}")

# --- Eksplorasi struktur data ---
print("\n" + "="*60)
print("STRUKTUR DATA (baris pertama split 'train')")
print("="*60)
sample = ds['train'][0]
print(f"\n📝 Context (200 karakter pertama):")
print(f"  {sample['context'][:200]}...")
print(f"\n❓ Jumlah QA pairs pada baris ini: {len(sample['qas'])}")
for i, qa in enumerate(sample['qas']):
    print(f"\n  QA-{i+1}:")
    print(f"    Question   : {qa['question']}")
    print(f"    Is Impossible: {qa['is_impossible']}")
    if qa['answers']:
        print(f"    Answer     : {qa['answers'][0]['text']}")
    else:
        print(f"    Answer     : (tidak ada — pertanyaan tidak bisa dijawab)")

# --- Statistik split ---
print("\n" + "="*60)
print("STATISTIK DATASET")
print("="*60)
for split_name in ['train', 'validation', 'test']:
    split = ds[split_name]
    total_qa = sum(len(row['qas']) for row in split)
    answerable = sum(
        sum(1 for qa in row['qas'] if not qa['is_impossible'])
        for row in split
    )
    print(f"  {split_name:12s}: {len(split):5d} konteks | "
          f"{total_qa:5d} total QA | {answerable:5d} bisa dijawab")


⏳ Memuat dataset IDK-MRC...


README.md: 0.00B [00:00, ?B/s]

train.json: 0.00B [00:00, ?B/s]

valid.json: 0.00B [00:00, ?B/s]

test.json: 0.00B [00:00, ?B/s]

Generating train split:   0%|          | 0/3659 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/358 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/378 [00:00<?, ? examples/s]


✅ Dataset berhasil dimuat!

DatasetDict({
    train: Dataset({
        features: ['qas', 'context'],
        num_rows: 3659
    })
    validation: Dataset({
        features: ['qas', 'context'],
        num_rows: 358
    })
    test: Dataset({
        features: ['qas', 'context'],
        num_rows: 378
    })
})

STRUKTUR DATA (baris pertama split 'train')

📝 Context (200 karakter pertama):
  Douwes Dekker terlahir di Pasuruan, Jawa Timur, pada tanggal 8 Oktober 1879, sebagaimana yang dia tulis pada riwayat hidup singkat saat mendaftar di Universitas Zurich, September 1913. Ayahnya, August...

❓ Jumlah QA pairs pada baris ini: 2

  QA-1:
    Question   : dimanakah  Dr. Ernest François Eugène Douwes Dekker dilahirkan?
    Is Impossible: False
    Answer     : Pasuruan, Jawa Timur

  QA-2:
    Question   : di manakah  Dr. Ernest François Eugène Douwes Dekker meninggal?
    Is Impossible: True
    Answer     : (tidak ada — pertanyaan tidak bisa dijawab)

STATISTIK DATASET
  train       :

## 🔧 Cell 4 — Preprocessing & Persiapan Data

In [ ]:
# ============================================================
# CELL 4: Preprocessing data
# Kita ekstrak QA pairs yang bisa dijawab (is_impossible=False)
# dari split validation untuk evaluasi.
# ============================================================

def extract_qa_pairs(dataset_split, include_impossible=False):
    """
    Ekstrak QA pairs dari satu split dataset.

    Returns:
        contexts (list): konteks per QA pair
        questions (list): pertanyaan
        answers (list): jawaban ground truth
        context_idx (list): indeks baris asli dalam split
    """
    contexts, questions, answers, ctx_indices = [], [], [], []

    for row_idx, row in enumerate(dataset_split):
        ctx = row['context']
        for qa in row['qas']:
            if qa['is_impossible']:
                if include_impossible:
                    contexts.append(ctx)
                    questions.append(qa['question'])
                    answers.append("")  # tidak ada jawaban
                    ctx_indices.append(row_idx)
                continue
            if not qa['answers']:
                continue
            contexts.append(ctx)
            questions.append(qa['question'])
            answers.append(qa['answers'][0]['text'])
            ctx_indices.append(row_idx)

    return contexts, questions, answers, ctx_indices


# --- Ekstrak dari validation split ---
val_gt_contexts, val_questions, val_answers, val_row_indices =     extract_qa_pairs(ds['validation'], include_impossible=False)

print(f"✅ Total QA pairs bisa dijawab (validation): {len(val_questions)}")
print(f"\nContoh QA pair:")
print(f"  Pertanyaan : {val_questions[0]}")
print(f"  Jawaban    : {val_answers[0]}")
print(f"  Konteks    : {val_gt_contexts[0][:120]}...")

# --- Juga ekstrak train untuk building corpus yang lebih besar ---
train_gt_contexts, train_questions, train_answers, train_row_indices =     extract_qa_pairs(ds['train'], include_impossible=False)

print(f"\n✅ Total QA pairs bisa dijawab (train): {len(train_questions)}")

# --- Statistik panjang teks ---
ctx_lengths  = [len(c.split()) for c in val_gt_contexts[:500]]
q_lengths    = [len(q.split()) for q in val_questions[:500]]
ans_lengths  = [len(a.split()) for a in val_answers[:500]]

print("\n" + "="*60)
print("STATISTIK PANJANG TEKS (validation)")
print("="*60)
print(f"  Konteks  — rata-rata: {np.mean(ctx_lengths):.1f} kata | "
      f"max: {max(ctx_lengths)} kata")
print(f"  Pertanyaan — rata-rata: {np.mean(q_lengths):.1f} kata | "
      f"max: {max(q_lengths)} kata")
print(f"  Jawaban  — rata-rata: {np.mean(ans_lengths):.1f} kata | "
      f"max: {max(ans_lengths)} kata")


✅ Total QA pairs bisa dijawab (validation): 382

Contoh QA pair:
  Pertanyaan : Apa kepanjangan dari GPS?
  Jawaban    : Global Positioning System
  Konteks    : Sistem Pemosisi Global [1] (bahasa Inggris: Global Positioning System (GPS)) adalah sistem untuk menentukan letak di per...

✅ Total QA pairs bisa dijawab (train): 5042

STATISTIK PANJANG TEKS (validation)
  Konteks  — rata-rata: 76.6 kata | max: 285 kata
  Pertanyaan — rata-rata: 5.9 kata | max: 16 kata
  Jawaban  — rata-rata: 5.8 kata | max: 43 kata


## 🗄️ Cell 5 — Bangun Corpus Retrieval

In [ ]:
# ============================================================
# CELL 5: Bangun corpus untuk retrieval
# Corpus = semua konteks unik dari train + validation + test
# Ini mensimulasikan knowledge base yang sesungguhnya
# ============================================================

print("⏳ Membangun corpus dari seluruh split...")

all_corpus_contexts = []
seen = set()

for split_name in ['train', 'validation', 'test']:
    for row in ds[split_name]:
        ctx = row['context'].strip()
        if ctx not in seen:
            seen.add(ctx)
            all_corpus_contexts.append(ctx)

print(f"\n✅ Corpus siap!")
print(f"   Total konteks unik : {len(all_corpus_contexts)}")

# Buat mapping konteks → indeks untuk pencarian cepat
corpus_ctx_to_idx = {ctx: i for i, ctx in enumerate(all_corpus_contexts)}

# Verifikasi: pastikan semua GT contexts ada dalam corpus
missing = sum(
    1 for ctx in val_gt_contexts
    if ctx not in corpus_ctx_to_idx
)
print(f"   GT contexts tidak ditemukan di corpus: {missing} "
      f"(harusnya 0 atau sangat kecil)")

# Contoh corpus
print(f"\nContoh konteks korpus ke-0:")
print(f"  {all_corpus_contexts[0][:200]}...")


⏳ Membangun corpus dari seluruh split...

✅ Corpus siap!
   Total konteks unik : 4395
   GT contexts tidak ditemukan di corpus: 0 (harusnya 0 atau sangat kecil)

Contoh konteks korpus ke-0:
  Douwes Dekker terlahir di Pasuruan, Jawa Timur, pada tanggal 8 Oktober 1879, sebagaimana yang dia tulis pada riwayat hidup singkat saat mendaftar di Universitas Zurich, September 1913. Ayahnya, August...


## 🧠 Cell 6 — Load Embedding Model (Sentence Transformer)

In [ ]:
# ============================================================
# CELL 6: Load embedding model
# paraphrase-multilingual-MiniLM-L12-v2:
#   - Mendukung 50+ bahasa termasuk Bahasa Indonesia
#   - Ringan (~420 MB) dan cepat
#   - Dimensi embedding: 384
# ============================================================

EMBEDDING_MODEL_NAME = 'sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2'

print(f"⏳ Memuat embedding model: {EMBEDDING_MODEL_NAME}")
embed_model = SentenceTransformer(EMBEDDING_MODEL_NAME)
print(f"✅ Embedding model dimuat!")
print(f"   Dimensi output       : {embed_model.get_sentence_embedding_dimension()}")
print(f"   Max sequence length  : {embed_model.max_seq_length}")

# Test embedding
test_sentence = "Dimana Douwes Dekker dilahirkan?"
test_emb = embed_model.encode([test_sentence])
print(f"\nTest embedding:")
print(f"  Input   : '{test_sentence}'")
print(f"  Shape   : {test_emb.shape}")
print(f"  Norm    : {np.linalg.norm(test_emb):.4f}")


⏳ Memuat embedding model: sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2


modules.json:   0%|          | 0.00/229 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/122 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

tokenizer_config.json:   0%|          | 0.00/526 [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/9.08M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/239 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

✅ Embedding model dimuat!
   Dimensi output       : 384
   Max sequence length  : 128

Test embedding:
  Input   : 'Dimana Douwes Dekker dilahirkan?'
  Shape   : (1, 384)
  Norm    : 3.4881


## 🔎 Cell 7 — Bangun FAISS Index

In [ ]:
# ============================================================
# CELL 7: Encode semua corpus contexts & bangun FAISS index
# Menggunakan IndexFlatIP (Inner Product) dengan normalisasi L2
# → efektif = cosine similarity
#
# Estimasi waktu: ~3-5 menit untuk ~4000 konteks di Colab
# ============================================================

EMBED_DIM  = embed_model.get_sentence_embedding_dimension()
BATCH_SIZE = 64

print(f"⏳ Encoding {len(all_corpus_contexts)} konteks...")
print(f"   Batch size : {BATCH_SIZE}")
print(f"   Dimensi    : {EMBED_DIM}")

corpus_embeddings_list = []
for i in tqdm(range(0, len(all_corpus_contexts), BATCH_SIZE),
              desc="Encoding corpus"):
    batch = all_corpus_contexts[i : i + BATCH_SIZE]
    embs  = embed_model.encode(batch,
                               convert_to_numpy=True,
                               show_progress_bar=False,
                               normalize_embeddings=True)  # L2-norm in-place
    corpus_embeddings_list.append(embs)

corpus_embeddings = np.vstack(corpus_embeddings_list).astype('float32')
print(f"\n✅ Encoding selesai! Shape: {corpus_embeddings.shape}")

# --- Bangun FAISS index ---
print("\n⏳ Membangun FAISS index (IndexFlatIP)...")
faiss_index = faiss.IndexFlatIP(EMBED_DIM)
faiss_index.add(corpus_embeddings)

print(f"✅ FAISS index siap!")
print(f"   Total vektor tersimpan : {faiss_index.ntotal}")
print(f"   Tipe index             : {type(faiss_index).__name__}")

# --- Free memory ---
del corpus_embeddings_list
gc.collect()


⏳ Encoding 4395 konteks...
   Batch size : 64
   Dimensi    : 384


Encoding corpus:   0%|          | 0/69 [00:00<?, ?it/s]


✅ Encoding selesai! Shape: (4395, 384)

⏳ Membangun FAISS index (IndexFlatIP)...
✅ FAISS index siap!
   Total vektor tersimpan : 4395
   Tipe index             : IndexFlatIP


602

## 🔍 Cell 8 — Fungsi Retrieval & Uji Coba

In [ ]:
# ============================================================
# CELL 8: Definisi fungsi retrieval dan uji coba
# ============================================================

def retrieve(query: str, k: int = 5):
    """
    Retrieve top-k konteks dari corpus berdasarkan cosine similarity.

    Args:
        query (str): Pertanyaan/query pengguna
        k (int): Jumlah konteks yang diambil

    Returns:
        retrieved_contexts (list[str]): Konteks yang diambil, diurutkan by similarity
        scores (list[float]): Cosine similarity scores (0-1)
        indices (list[int]): Indeks dalam corpus
    """
    # Encode query
    q_emb = embed_model.encode(
        [query],
        convert_to_numpy=True,
        normalize_embeddings=True,
        show_progress_bar=False
    ).astype('float32')

    # Search FAISS
    scores, indices = faiss_index.search(q_emb, k)

    retrieved = [all_corpus_contexts[i] for i in indices[0]]
    return retrieved, scores[0].tolist(), indices[0].tolist()


# --- Uji coba retrieval ---
test_query = val_questions[0]
test_gt    = val_gt_contexts[0]

print(f"🔍 Test Retrieval")
print(f"{'='*60}")
print(f"Pertanyaan : {test_query}")
print(f"Ground Truth Answer: {val_answers[0]}")

for k_test in [1, 3, 5]:
    ret_ctxs, ret_scores, ret_idxs = retrieve(test_query, k=k_test)
    gt_found = any(c == test_gt for c in ret_ctxs)
    print(f"\nK={k_test}: GT ditemukan = {gt_found} | "
          f"Scores: {[f'{s:.3f}' for s in ret_scores]}")
    if k_test == 3:
        for i, (ctx, sc) in enumerate(zip(ret_ctxs, ret_scores)):
            marker = "✅" if ctx == test_gt else "  "
            print(f"  {marker} [{i+1}] (score={sc:.3f}) {ctx[:80]}...")


🔍 Test Retrieval
Pertanyaan : Apa kepanjangan dari GPS?
Ground Truth Answer: Global Positioning System

K=1: GT ditemukan = True | Scores: ['0.752']

K=3: GT ditemukan = True | Scores: ['0.752', '0.639', '0.574']
  ✅ [1] (score=0.752) Sistem Pemosisi Global [1] (bahasa Inggris: Global Positioning System (GPS)) ada...
     [2] (score=0.639) Kompas adalah alat navigasi untuk menentukan arah berupa sebuah panah penunjuk m...
     [3] (score=0.574) Linguatronik adalah sistem yang dilengkapi dengan GPS untuk menunjukkan alamat t...

K=5: GT ditemukan = True | Scores: ['0.752', '0.639', '0.574', '0.537', '0.499']


## 🤖 Cell 9 — Load LLM dengan 4-bit Quantization

In [ ]:
# ============================================================
# CELL 9: Load Qwen2.5-1.5B-Instruct dengan bitsandbytes 4-bit
#
# Model ini dipilih karena:
#   - Hanya ~1.5B parameter → muat di T4 (15 GB VRAM) dengan 4-bit
#   - Instruction-tuned → mengikuti format instruksi dengan baik
#   - Mendukung teks Indonesia cukup baik
#
# Alternatiif jika OOM: google/gemma-2b-it
# ============================================================

LLM_MODEL_NAME = "Qwen/Qwen2.5-1.5B-Instruct"
# LLM_MODEL_NAME = "google/gemma-2b-it"  # ← uncomment jika prefer Gemma

print(f"⏳ Memuat LLM: {LLM_MODEL_NAME}")
print(f"   Konfigurasi: 4-bit NF4 quantization via bitsandbytes")

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_use_double_quant=True,
)

llm_tokenizer = AutoTokenizer.from_pretrained(
    LLM_MODEL_NAME,
    trust_remote_code=True,
)

llm_model = AutoModelForCausalLM.from_pretrained(
    LLM_MODEL_NAME,
    quantization_config=bnb_config,
    device_map="auto",
    trust_remote_code=True,
)
llm_model.eval()

# Pastikan pad_token tersedia
if llm_tokenizer.pad_token is None:
    llm_tokenizer.pad_token = llm_tokenizer.eos_token

print(f"\n✅ LLM berhasil dimuat!")
print(f"   Tipe model     : {type(llm_model).__name__}")
print(f"   Device map     : {llm_model.hf_device_map if hasattr(llm_model, 'hf_device_map') else 'auto'}")

if torch.cuda.is_available():
    allocated = torch.cuda.memory_allocated() / 1e9
    reserved  = torch.cuda.memory_reserved()  / 1e9
    print(f"   VRAM digunakan : {allocated:.2f} GB allocated / {reserved:.2f} GB reserved")


⏳ Memuat LLM: Qwen/Qwen2.5-1.5B-Instruct
   Konfigurasi: 4-bit NF4 quantization via bitsandbytes


config.json:   0%|          | 0.00/660 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/3.09G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/242 [00:00<?, ?B/s]


✅ LLM berhasil dimuat!
   Tipe model     : Qwen2ForCausalLM
   Device map     : auto
   VRAM digunakan : 1.64 GB allocated / 1.67 GB reserved


## 💬 Cell 10 — Prompt Template & Fungsi Generasi

In [ ]:
# ============================================================
# CELL 10: Desain prompt dan fungsi generasi jawaban
#
# Desain prompt RAG:
#  1. Sistem menunjukkan dokumen bernomor [Dokumen 1], [Dokumen 2], ...
#  2. LLM diminta menjawab singkat dan MENYEBUTKAN nomor dokumen
#     yang dijadikan referensi → ini kita gunakan untuk mengukur
#     akurasi sitasi (citation accuracy)
# ============================================================

SYSTEM_PROMPT = (
    "Anda adalah asisten cerdas yang menjawab pertanyaan berdasarkan "
    "dokumen referensi yang diberikan. Jawab secara singkat dan tepat "
    "dalam Bahasa Indonesia. Selalu sebutkan nomor dokumen yang Anda "
    "gunakan sebagai referensi."
)


def build_rag_prompt(question: str, contexts: list[str]) -> str:
    """
    Bangun prompt RAG dengan konteks bernomor untuk mendukung sitasi.

    Format:
        [Dokumen 1]
        <isi konteks 1>

        [Dokumen 2]
        <isi konteks 2>
        ...

        Pertanyaan: <pertanyaan>

        Instruksi:
        1. Jawab berdasarkan dokumen di atas.
        2. Tulis [Referensi: Dokumen X] di akhir jawaban.
    """
    doc_block = ""
    for i, ctx in enumerate(contexts, 1):
        doc_block += f"[Dokumen {i}]\n{ctx}\n\n"

    prompt = (
        f"Berikut adalah dokumen referensi:\n\n"
        f"{doc_block}"
        f"Pertanyaan: {question}\n\n"
        f"Instruksi:\n"
        f"1. Jawab pertanyaan secara singkat dan tepat berdasarkan dokumen di atas.\n"
        f"2. Di akhir jawaban, tambahkan format berikut:\n"
        f"   [Referensi: Dokumen X] — jika menggunakan satu dokumen\n"
        f"   [Referensi: Dokumen X, Dokumen Y] — jika menggunakan lebih dari satu\n"
        f"   [Referensi: Tidak ada] — jika tidak ada dokumen yang relevan\n\n"
        f"Jawaban:"
    )
    return prompt


def generate_answer(question: str,
                    contexts: list[str],
                    max_new_tokens: int = 256) -> str:
    """
    Generate jawaban menggunakan LLM dengan konteks RAG.

    Args:
        question: Pertanyaan yang akan dijawab
        contexts: Daftar konteks hasil retrieval
        max_new_tokens: Maksimum token yang di-generate

    Returns:
        answer (str): Jawaban yang di-generate oleh LLM
    """
    prompt_text = build_rag_prompt(question, contexts)

    messages = [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user",   "content": prompt_text},
    ]

    # Apply chat template
    text = llm_tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True
    )

    # Tokenize
    model_inputs = llm_tokenizer(
        [text],
        return_tensors="pt",
        truncation=True,
        max_length=2048,
    ).to(llm_model.device)

    # Generate
    with torch.no_grad():
        output_ids = llm_model.generate(
            **model_inputs,
            max_new_tokens=max_new_tokens,
            do_sample=False,           # greedy decoding → deterministic
            temperature=1.0,
            repetition_penalty=1.1,
            pad_token_id=llm_tokenizer.eos_token_id,
        )

    # Decode hanya bagian baru (after prompt)
    new_token_ids = output_ids[0][model_inputs['input_ids'].shape[1]:]
    answer = llm_tokenizer.decode(new_token_ids, skip_special_tokens=True)

    return answer.strip()


# --- Uji coba generasi ---
print("🧪 Uji coba generasi jawaban (K=3)...")
demo_q   = val_questions[0]
demo_ans = val_answers[0]
demo_ctxs, _, _ = retrieve(demo_q, k=3)

demo_output = generate_answer(demo_q, demo_ctxs)

print(f"\nPertanyaan  : {demo_q}")
print(f"Ground Truth: {demo_ans}")
print(f"\nJawaban LLM :\n{demo_output}")


🧪 Uji coba generasi jawaban (K=3)...

Pertanyaan  : Apa kepanjangan dari GPS?
Ground Truth: Global Positioning System

Jawaban LLM :
GPS adalah singkatan dari Global Positioning System.

[Referensi: Dokumen 1]


## 📌 Cell 11 — Ekstraksi Sitasi & Citation Accuracy

In [ ]:
# ============================================================
# CELL 11: Fungsi untuk mengekstrak nomor dokumen yang disitasi
# oleh LLM, dan menghitung citation accuracy
#
# Citation accuracy = persentase kasus di mana LLM menyitasi
# dokumen yang benar-benar mengandung jawaban ground truth
# ============================================================

def extract_cited_doc_numbers(answer_text: str) -> list[int]:
    """
    Ekstrak nomor dokumen yang disitasi dari teks jawaban LLM.
    Menangani berbagai variasi format output.

    Returns:
        list[int]: nomor-nomor dokumen yang disitasi (1-indexed)
    """
    cited = set()

    # Pola 1: [Referensi: Dokumen 1, Dokumen 2]
    pattern1 = r'\[Referensi:\s*Dokumen\s*(\d+(?:[,\s]+Dokumen\s*\d+)*)\]'
    matches1 = re.findall(pattern1, answer_text, re.IGNORECASE)
    for m in matches1:
        nums = re.findall(r'\d+', m)
        cited.update(int(n) for n in nums)

    # Pola 2: [Referensi: 1, 2]
    pattern2 = r'\[Referensi:\s*(\d+(?:[,\s]+\d+)*)\]'
    matches2 = re.findall(pattern2, answer_text, re.IGNORECASE)
    for m in matches2:
        nums = re.findall(r'\d+', m)
        cited.update(int(n) for n in nums)

    # Pola 3: Referensi: Dokumen X (tanpa kurung siku)
    pattern3 = r'Referensi:\s*Dokumen\s*(\d+(?:[,\s]+(?:Dokumen\s*)?\d+)*)'
    matches3 = re.findall(pattern3, answer_text, re.IGNORECASE)
    for m in matches3:
        nums = re.findall(r'\d+', m)
        cited.update(int(n) for n in nums)

    # Pola 4: (Dokumen 1) inline
    pattern4 = r'\(Dokumen\s*(\d+)\)'
    matches4 = re.findall(pattern4, answer_text, re.IGNORECASE)
    cited.update(int(n) for n in matches4)

    # Pola 5: Dokumen 1 tanpa tanda apapun (fallback agresif)
    if not cited:
        pattern5 = r'[Dd]okumen\s*(\d+)'
        matches5 = re.findall(pattern5, answer_text)
        cited.update(int(n) for n in matches5)

    return sorted(list(cited))


def compute_citation_accuracy(
    cited_doc_numbers: list[int],
    ground_truth_context: str,
    retrieved_contexts: list[str]
) -> tuple[int | None, bool]:
    """
    Hitung citation accuracy untuk satu sampel.

    Args:
        cited_doc_numbers : nomor dokumen yang disitasi LLM (1-indexed)
        ground_truth_context : konteks yang benar-benar mengandung jawaban
        retrieved_contexts : daftar konteks yang di-retrieve (urut)

    Returns:
        accuracy (int | None): 1 = benar, 0 = salah, None = GT tidak ada dalam retrieved
        gt_in_retrieved (bool): apakah GT ada dalam retrieved contexts
    """
    # Cari posisi GT dalam retrieved contexts (bisa lebih dari 1 karena duplikat)
    gt_positions = [
        i + 1  # 1-indexed
        for i, ctx in enumerate(retrieved_contexts)
        if ctx == ground_truth_context
    ]

    gt_in_retrieved = len(gt_positions) > 0

    if not gt_in_retrieved:
        # Ground truth tidak ada dalam retrieved → tidak bisa dievaluasi sitasi
        return None, False

    if not cited_doc_numbers:
        # LLM tidak menyitasi apapun → salah
        return 0, True

    # Cek apakah ada nomor yang disitasi cocok dengan posisi GT
    for cited in cited_doc_numbers:
        if cited in gt_positions:
            return 1, True

    return 0, True


# --- Uji coba ---
test_answers_examples = [
    "Dr. Douwes Dekker lahir di Pasuruan. [Referensi: Dokumen 1]",
    "Beliau lahir di Pasuruan, Jawa Timur [Referensi: Dokumen 2, Dokumen 3]",
    "Saya tidak tahu jawabannya. [Referensi: Tidak ada]",
    "Dilahirkan di Pasuruan menurut Dokumen 1.",
]
print("🧪 Uji coba ekstraksi sitasi:")
for ex in test_answers_examples:
    cited = extract_cited_doc_numbers(ex)
    print(f"  Input  : {ex[:70]}")
    print(f"  Sitasi : {cited}\n")


🧪 Uji coba ekstraksi sitasi:
  Input  : Dr. Douwes Dekker lahir di Pasuruan. [Referensi: Dokumen 1]
  Sitasi : [1]

  Input  : Beliau lahir di Pasuruan, Jawa Timur [Referensi: Dokumen 2, Dokumen 3]
  Sitasi : [2, 3]

  Input  : Saya tidak tahu jawabannya. [Referensi: Tidak ada]
  Sitasi : []

  Input  : Dilahirkan di Pasuruan menurut Dokumen 1.
  Sitasi : [1]



## 📊 Cell 12 — Fungsi Metrik Evaluasi

In [ ]:
# ============================================================
# CELL 12: Implementasi metrik evaluasi
#   - ROUGE-1, ROUGE-2, ROUGE-L : text overlap n-gram
#   - BLEU                       : precision-based n-gram metric
#   - BERTScore                  : semantic similarity (computed later)
# ============================================================

# Inisialisasi ROUGE scorer
rouge_eval = rouge_scorer.RougeScorer(
    ['rouge1', 'rouge2', 'rougeL'],
    use_stemmer=False  # jangan stemming → lebih cocok untuk Bahasa Indonesia
)

def compute_rouge_scores(prediction: str, reference: str) -> dict:
    """
    Hitung ROUGE-1, ROUGE-2, ROUGE-L F1 antara prediksi dan referensi.

    Args:
        prediction: Teks yang di-generate oleh model
        reference : Teks ground truth

    Returns:
        dict dengan kunci 'rouge1', 'rouge2', 'rougeL' (float, 0-1)
    """
    if not prediction.strip() or not reference.strip():
        return {'rouge1': 0.0, 'rouge2': 0.0, 'rougeL': 0.0}

    scores = rouge_eval.score(reference, prediction)
    return {
        'rouge1': scores['rouge1'].fmeasure,
        'rouge2': scores['rouge2'].fmeasure,
        'rougeL': scores['rougeL'].fmeasure,
    }


def compute_bleu_score(prediction: str, reference: str) -> float:
    """
    Hitung BLEU score menggunakan smoothing method4 (Chen & Cherry 2014).
    Smoothing diperlukan karena jawaban seringkali sangat pendek.

    Args:
        prediction: Teks yang di-generate
        reference : Teks ground truth

    Returns:
        bleu (float): BLEU score (0-1)
    """
    if not prediction.strip() or not reference.strip():
        return 0.0

    ref_tokens  = reference.lower().split()
    hyp_tokens  = prediction.lower().split()

    if len(hyp_tokens) == 0:
        return 0.0

    try:
        smoothing = SmoothingFunction().method4
        score = sentence_bleu(
            [ref_tokens],
            hyp_tokens,
            weights=(0.25, 0.25, 0.25, 0.25),
            smoothing_function=smoothing
        )
    except Exception:
        score = 0.0

    return float(score)


# --- Uji coba metrik ---
pred_ex = "Dr. Douwes Dekker dilahirkan di Pasuruan, Jawa Timur"
ref_ex  = "Pasuruan, Jawa Timur"

rouge_ex = compute_rouge_scores(pred_ex, ref_ex)
bleu_ex  = compute_bleu_score(pred_ex, ref_ex)

print("🧪 Uji coba metrik:")
print(f"  Prediksi  : '{pred_ex}'")
print(f"  Referensi : '{ref_ex}'")
print(f"  ROUGE-1   : {rouge_ex['rouge1']:.4f}")
print(f"  ROUGE-2   : {rouge_ex['rouge2']:.4f}")
print(f"  ROUGE-L   : {rouge_ex['rougeL']:.4f}")
print(f"  BLEU      : {bleu_ex:.4f}")


🧪 Uji coba metrik:
  Prediksi  : 'Dr. Douwes Dekker dilahirkan di Pasuruan, Jawa Timur'
  Referensi : 'Pasuruan, Jawa Timur'
  ROUGE-1   : 0.5455
  ROUGE-2   : 0.4444
  ROUGE-L   : 0.5455
  BLEU      : 0.1651


## 🧪 Cell 13 — Main Experiment Loop (K = 1, 3, 5, 10)

In [ ]:
# ============================================================
# CELL 13: Loop eksperimen utama
#
# Untuk setiap nilai K dalam {1, 3, 5, 10}:
#   1. Retrieve K konteks per pertanyaan
#   2. Generate jawaban dengan LLM
#   3. Hitung ROUGE, BLEU
#   4. Ekstrak sitasi & hitung citation accuracy
#
# NUM_SAMPLES: jumlah sampel yang dievaluasi
#   → 100 sudah cukup representatif dan selesai dalam ~30 menit
#   → Naikkan ke 200-358 untuk hasil lebih akurat (lebih lama)
# ============================================================

K_VALUES    = [1, 3, 5, 10]
NUM_SAMPLES = 100           # ← Bisa dinaikkan untuk hasil lebih akurat

# Sampel acak dari validation set
np.random.seed(42)
sample_idx = np.random.choice(
    len(val_questions),
    size=min(NUM_SAMPLES, len(val_questions)),
    replace=False
)

sampled_questions   = [val_questions[i]    for i in sample_idx]
sampled_answers     = [val_answers[i]      for i in sample_idx]
sampled_gt_contexts = [val_gt_contexts[i]  for i in sample_idx]

print(f"📋 Konfigurasi Eksperimen")
print(f"{'='*50}")
print(f"  K values     : {K_VALUES}")
print(f"  Jumlah sampel: {len(sampled_questions)}")
print(f"  LLM          : {LLM_MODEL_NAME}")
print(f"  Embedding    : {EMBEDDING_MODEL_NAME}")
print(f"  Dataset      : IDK-MRC (validation split)")
print()

# Simpan semua hasil
all_results = {}

TOTAL_RUNS = len(K_VALUES) * len(sampled_questions)
run_count  = 0
start_time = time.time()

for K in K_VALUES:
    print(f"\n{'='*60}")
    print(f"▶ Eksperimen K = {K}")
    print(f"{'='*60}")

    k_data = {
        'K'                 : K,
        'questions'         : [],
        'ground_truths'     : [],
        'generated_answers' : [],
        'rouge1'            : [],
        'rouge2'            : [],
        'rougeL'            : [],
        'bleu'              : [],
        'citation_correct'  : [],  # 1=benar, 0=salah
        'gt_in_retrieved'   : [],  # True/False
        'cited_docs'        : [],  # nomor dokumen yang disitasi
        'n_citation_evaluable': 0, # sampel yang bisa dievaluasi sitasinya
    }

    for i, (q, gt_ans, gt_ctx) in enumerate(
        tqdm(
            zip(sampled_questions, sampled_answers, sampled_gt_contexts),
            total=len(sampled_questions),
            desc=f"K={K}"
        )
    ):
        try:
            # ── 1. Retrieve ──────────────────────────────────────
            ret_ctxs, ret_scores, ret_idxs = retrieve(q, k=K)

            # ── 2. Generate ──────────────────────────────────────
            generated = generate_answer(q, ret_ctxs, max_new_tokens=256)

            # ── 3. ROUGE & BLEU ──────────────────────────────────
            rouge = compute_rouge_scores(generated, gt_ans)
            bleu  = compute_bleu_score(generated, gt_ans)

            # ── 4. Citation accuracy ─────────────────────────────
            cited_nums = extract_cited_doc_numbers(generated)
            cit_acc, gt_in_ret = compute_citation_accuracy(
                cited_nums, gt_ctx, ret_ctxs
            )

            # ── Simpan ───────────────────────────────────────────
            k_data['questions'].append(q)
            k_data['ground_truths'].append(gt_ans)
            k_data['generated_answers'].append(generated)
            k_data['rouge1'].append(rouge['rouge1'])
            k_data['rouge2'].append(rouge['rouge2'])
            k_data['rougeL'].append(rouge['rougeL'])
            k_data['bleu'].append(bleu)
            k_data['gt_in_retrieved'].append(gt_in_ret)
            k_data['cited_docs'].append(cited_nums)

            if cit_acc is not None:
                k_data['citation_correct'].append(cit_acc)
                k_data['n_citation_evaluable'] += 1

        except Exception as e:
            print(f"  ⚠️  Error pada sampel {i}: {e}")
            # Isi dengan nilai default supaya indeks tetap sinkron
            k_data['questions'].append(q)
            k_data['ground_truths'].append(gt_ans)
            k_data['generated_answers'].append("")
            k_data['rouge1'].append(0.0)
            k_data['rouge2'].append(0.0)
            k_data['rougeL'].append(0.0)
            k_data['bleu'].append(0.0)
            k_data['gt_in_retrieved'].append(False)
            k_data['cited_docs'].append([])
            continue

        run_count += 1

    all_results[K] = k_data

    # Print ringkasan sementara
    cit_acc_mean = (
        np.mean(k_data['citation_correct'])
        if k_data['citation_correct'] else float('nan')
    )
    print(f"\n📊 Ringkasan K={K}:")
    print(f"   ROUGE-1  : {np.mean(k_data['rouge1']):.4f}")
    print(f"   ROUGE-2  : {np.mean(k_data['rouge2']):.4f}")
    print(f"   ROUGE-L  : {np.mean(k_data['rougeL']):.4f}")
    print(f"   BLEU     : {np.mean(k_data['bleu']):.4f}")
    print(f"   GT in Retrieved : {np.mean(k_data['gt_in_retrieved'])*100:.1f}%")
    print(f"   Citation Accuracy: {cit_acc_mean:.4f} "
          f"(n={k_data['n_citation_evaluable']})")

elapsed = time.time() - start_time
print(f"\n✅ Eksperimen selesai! Total waktu: {elapsed/60:.1f} menit")


📋 Konfigurasi Eksperimen
  K values     : [1, 3, 5, 10]
  Jumlah sampel: 100
  LLM          : Qwen/Qwen2.5-1.5B-Instruct
  Embedding    : sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2
  Dataset      : IDK-MRC (validation split)


▶ Eksperimen K = 1


K=1:   0%|          | 0/100 [00:00<?, ?it/s]


📊 Ringkasan K=1:
   ROUGE-1  : 0.1842
   ROUGE-2  : 0.1028
   ROUGE-L  : 0.1754
   BLEU     : 0.0431
   GT in Retrieved : 55.0%
   Citation Accuracy: 0.9273 (n=55)

▶ Eksperimen K = 3


K=3:   0%|          | 0/100 [00:00<?, ?it/s]


📊 Ringkasan K=3:
   ROUGE-1  : 0.2050
   ROUGE-2  : 0.1289
   ROUGE-L  : 0.1979
   BLEU     : 0.0607
   GT in Retrieved : 72.0%
   Citation Accuracy: 0.8889 (n=72)

▶ Eksperimen K = 5


K=5:   0%|          | 0/100 [00:00<?, ?it/s]


📊 Ringkasan K=5:
   ROUGE-1  : 0.2011
   ROUGE-2  : 0.1243
   ROUGE-L  : 0.1955
   BLEU     : 0.0565
   GT in Retrieved : 80.0%
   Citation Accuracy: 0.8375 (n=80)

▶ Eksperimen K = 10


K=10:   0%|          | 0/100 [00:00<?, ?it/s]


📊 Ringkasan K=10:
   ROUGE-1  : 0.1040
   ROUGE-2  : 0.0701
   ROUGE-L  : 0.1019
   BLEU     : 0.0428
   GT in Retrieved : 86.0%
   Citation Accuracy: 0.3140 (n=86)

✅ Eksperimen selesai! Total waktu: 51.0 menit


## 🤗 Cell 14 — Hitung BERTScore (Semantic Similarity)

In [ ]:
# ============================================================
# CELL 14: Hitung BERTScore untuk semua K values
#
# BERTScore mengukur semantic similarity menggunakan representasi
# kontekstual dari BERT → lebih robust dari ROUGE/BLEU untuk
# kasus sinonimia atau parafrase.
#
# Model yang digunakan: bert-base-multilingual-cased
# → mendukung Bahasa Indonesia
# ============================================================

print("⏳ Menghitung BERTScore untuk semua K values...")
print("   Model: bert-base-multilingual-cased")
print("   (Proses ini membutuhkan ~2-5 menit)")
print()

bert_model_type = 'bert-base-multilingual-cased'

for K in K_VALUES:
    preds = all_results[K]['generated_answers']
    refs  = all_results[K]['ground_truths']

    # Filter pasangan kosong
    valid_pairs = [
        (p, r) for p, r in zip(preds, refs)
        if p.strip() and r.strip()
    ]

    if not valid_pairs:
        all_results[K]['bertscore_f1'] = [0.0] * len(preds)
        continue

    valid_preds, valid_refs = zip(*valid_pairs)

    print(f"  K={K}: Evaluasi {len(valid_preds)} pasang teks...")

    try:
        P, R, F1 = bert_score_fn(
            list(valid_preds),
            list(valid_refs),
            model_type=bert_model_type,
            lang='id',
            verbose=False,
            batch_size=16,
        )
        f1_scores = F1.tolist()
    except Exception as e:
        print(f"  ⚠️  BERTScore error pada K={K}: {e}")
        f1_scores = [0.0] * len(valid_preds)

    # Kembalikan ke panjang asli (isi 0 untuk yang kosong)
    full_f1 = []
    valid_iter = iter(f1_scores)
    for p, r in zip(preds, refs):
        if p.strip() and r.strip():
            full_f1.append(next(valid_iter))
        else:
            full_f1.append(0.0)

    all_results[K]['bertscore_f1'] = full_f1
    print(f"     BERTScore F1 (avg): {np.mean(f1_scores):.4f}")

print("\n✅ BERTScore selesai!")


⏳ Menghitung BERTScore untuk semua K values...
   Model: bert-base-multilingual-cased
   (Proses ini membutuhkan ~2-5 menit)

  K=1: Evaluasi 100 pasang teks...


config.json:   0%|          | 0.00/625 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/49.0 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

...

[transformers] BertModel LOAD REPORT from: bert-base-multilingual-cased
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.seq_relationship.weight                | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 
cls.predictions.bias                       | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 
cls.seq_relationship.bias                  | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


     BERTScore F1 (avg): 0.6500
  K=10: Evaluasi 100 pasang teks...


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

[transformers] BertModel LOAD REPORT from: bert-base-multilingual-cased
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.seq_relationship.weight                | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 
cls.predictions.bias                       | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 
cls.seq_relationship.bias                  | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


     BERTScore F1 (avg): 0.5779

✅ BERTScore selesai!
